# AgentCore Runtime でのランタイムコンテキストとセッション管理の理解

## 概要

このチュートリアルでは、Amazon Bedrock AgentCore Runtime でのランタイムコンテキストとセッション管理を理解し、操作する方法を学習します。この例では、AgentCore Runtime がセッションを処理し、複数の呼び出しにわたってコンテキストを維持し、エージェントがコンテキストオブジェクトを通じてランタイム情報にアクセスする方法を示します。

Amazon Bedrock AgentCore Runtime は、各ユーザーインタラクションに対して分離されたセッションを提供し、異なるユーザー間で完全なセキュリティ分離を確保しながら、エージェントが複数の呼び出しにわたってコンテキストと状態を維持できるようにします。

### チュートリアルの詳細

|情報| 詳細|
|:--------------------|:---------------------------------------------------------------------------------|
| チュートリアルタイプ       | コンテキストとセッション管理|
| エージェントタイプ          | 単一         |
| エージェントフレームワーク   | Strands Agents |
| LLM モデル           | Anthropic Claude Haiku 4.5 |
| チュートリアルコンポーネント | ランタイムコンテキスト、セッション管理、AgentCore Runtime、Strands Agent と Amazon Bedrock Model |
| チュートリアル垂直領域   | クロス垂直                                                                   |
| 例の複雑さ  | 中級                                                                     |
| 使用するSDK            | Amazon BedrockAgentCore Python SDK と boto3|

### チュートリアルアーキテクチャ

このチュートリアルでは、Amazon Bedrock AgentCore Runtime がセッションを管理し、エージェントにコンテキストを提供する方法を探ります。以下を実演します：

1. **セッション継続性**: 同じセッション ID が複数の呼び出しにわたってコンテキストを維持する方法
2. **コンテキストオブジェクト**: エージェントがコンテキストパラメータを通じてランタイム情報にアクセスする方法
3. **セッション分離**: 異なるセッション ID が完全に分離された環境を作成する方法
4. **ペイロードの柔軟性**: ペイロードを通じてカスタムデータをエージェントに渡す方法

デモンストレーションの目的で、これらのセッション管理機能を示す Strands Agent を使用します。

    
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>

### チュートリアルの主な機能

* **セッションベースのコンテキスト管理**: AgentCore Runtime がセッション内でコンテキストを維持する方法の理解
* **ランタイムセッションライフサイクル**: セッションの作成、維持、終了についての学習
* **コンテキストオブジェクトへのアクセス**: コンテキストパラメータを通じてセッション ID などのランタイム情報にアクセス
* **セッション分離**: 異なるセッションが完全な分離を提供する方法の実演
* **ペイロード処理**: カスタムペイロード構造を通じた柔軟なデータ受け渡し
* **呼び出し間の状態**: 同じセッション内の複数の呼び出しにわたるエージェント状態の維持

## 前提条件

このチュートリアルを実行するには、以下が必要です：
* Python 3.10+
* AWS 認証情報
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker が実行中

## Amazon Bedrock AgentCore Runtime セッションの理解

コードに入る前に、Amazon Bedrock AgentCore Runtime がセッションを管理する方法を理解することが重要です：

### セッション分離とセキュリティ

AgentCore Runtime は、専用の microVM を通じて**完全なセッション分離**を提供します：

- **専用リソース**: 各セッションは、分離された CPU、メモリ、ファイルシステムを持つ独自の microVM で実行される
- **セキュリティ境界**: ユーザーセッション間の完全な分離により、データ汚染を防止
- **決定論的クリーンアップ**: セッション完了後、microVM は終了し、メモリはサニタイズされる

### セッションライフサイクル

AgentCore Runtime のセッションは、特定のライフサイクルに従います：

1. **作成**: 一意の `runtimeSessionId` で最初の呼び出し時に新しいセッションが作成される
2. **アクティブ状態**: セッションはリクエストを処理し、コンテキストを維持する
3. **アイドル状態**: セッションはコンテキストを保持しながら次の呼び出しを待つ
4. **終了**: セッションは以下の理由で終了する：
   - 非アクティブ（15分）
   - 最大ライフタイム（8時間）
   - ヘルスチェックの失敗

### コンテキストの永続化

セッション内で、AgentCore Runtime は以下を維持します：
- **会話履歴**: 以前のインタラクションと応答
- **アプリケーション状態**: 実行中に作成された変数とオブジェクト
- **ファイルシステム**: セッション中に作成または変更されたファイル
- **環境変数**: カスタム設定と構成

### セッション管理のベストプラクティス

- **一意のセッション ID**: 各ユーザーまたは会話に対して一意のセッション ID を生成
- **コンテキストの再利用**: 関連する呼び出しには同じセッション ID を使用してコンテキストを維持
- **セッション境界**: 異なるユーザーまたは無関係な会話には異なるセッション ID を使用
- **一時的な性質**: 永続的なデータストレージにはセッションに依存しない（永続化には AgentCore Memory を使用）

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## AgentCore Runtime へのデプロイ用のエージェントの準備

セッション管理とコンテキスト処理を実演するために、エージェントを AgentCore Runtime にデプロイしましょう。エージェントは以下を示します：

1. **ランタイムコンテキストへのアクセス**: `context` パラメータを使用してセッション情報を取得
2. **カスタムペイロードの処理**: ペイロードを通じて渡される構造化データを処理
3. **セッション状態の維持**: セッション内のユーザーインタラクションを追跡
4. **セッション境界の実演**: 異なるセッションがどのように分離されるかを示す

### コンテキストオブジェクトの理解

AgentCore Runtime の `context` オブジェクトは、現在の実行環境に関する貴重な情報を提供します：

- **session_id**: 現在のランタイムセッション識別子
- **ランタイムメタデータ**: ランタイム環境に関する情報
- **実行詳細**: 現在の呼び出しに関するコンテキスト

### コンテキスト処理を備えた Strands Agent

セッション管理とコンテキスト処理を示す実装を見てみましょう：

In [ ]:
%%writefile strands_claude_context.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel
import asyncio
from datetime import datetime

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

@tool
def get_time():
    """ Get current time """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[
        calculator, weather, get_time
    ],
    system_prompt="""
    You're a helpful assistant. You can do simple math calculations, 
    tell the weather, and provide the current time.
    Always start by acknowledging the user's name 
    """
)

def get_user_name(user_id):
    users = {
        "1": "Maira",
        "2": "Mani",
        "3": "Mark",
        "4": "Ishan",
        "5": "Dhawal"
    }
    return users[user_id]
    
@app.entrypoint
def strands_agent_bedrock_handling_context(payload, context):
    """
    AgentCore Runtime entrypoint that demonstrates context handling and session management.
    
    Args:
        payload: The input payload containing user data and request information
        context: The runtime context object containing session and execution information
    
    Returns:
        str: The agent's response incorporating context information
    """
    user_input = payload.get("prompt")
    user_id = payload.get("user_id")
    user_name = get_user_name(user_id)
    
    # Access runtime context information
    print("=== Runtime Context Information ===")
    print("User id:", user_id)
    print("User Name:", user_name)
    print("User input:", user_input)
    print("Runtime Session ID:", context.session_id)
    print("Context Object Type:", type(context))
    print("=== End Context Information ===")
    
    # Create a personalized prompt that includes context information
    prompt = f"""My name is {user_name}. Here is my request: {user_input}
    
    Additional context: This is session {context.session_id}. 
    Please acknowledge my name and provide assistance."""
    
    response = agent(prompt)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## AgentCore Runtime でのセッション管理の理解

上記のコードは、AgentCore Runtime がセッションを管理し、エージェントにコンテキストを提供する方法に関するいくつかの重要な概念を示しています：

### コンテキストオブジェクト構造

エントリーポイント関数の `context` パラメータは、ランタイム情報へのアクセスを提供します：

```python
@app.entrypoint
def strands_agent_bedrock_handling_context(payload, context):
    # セッション情報にアクセス
    session_id = context.session_id
    # エージェントロジックでコンテキスト情報を使用
```

### セッション継続性の利点

単一のセッション内で、AgentCore Runtime は以下を提供します：

1. **永続的な環境**: 変数と状態が呼び出し間で永続化される
2. **コンテキストの保持**: エージェントが以前のインタラクションを参照できる
3. **リソースの再利用**: 初期化されたモデルとツールが読み込まれたまま
4. **パフォーマンスの利点**: 後続の呼び出しでのコールドスタート時間の短縮

### セッション分離の保証

AgentCore Runtime は、セッション間の完全な分離を保証します：

- **セキュリティ**: 各セッションは分離されたリソースを持つ独自の microVM で実行される
- **プライバシー**: 異なるユーザーセッション間でデータ漏洩がない
- **信頼性**: 1つのセッションの問題が他のセッションに影響しない
- **クリーンアップ**: セッション終了後の完全なメモリサニタイゼーション

### ペイロードの柔軟性

`payload` パラメータは柔軟なデータ受け渡しを可能にします：

```python
# ペイロード構造の例
payload = {
    "prompt": "ユーザーの質問",
    "user_id": "1",
    "preferences": {...},
    "context_data": {...}
}
```

これにより、ランタイムによって提供されるセッションコンテキストを維持しながら、クライアントとエージェント間のリッチで構造化された通信が可能になります。

### AgentCore Runtime デプロイの設定

次に、スターターツールキットを使用して、エントリーポイント、作成した実行ロール、requirements ファイルで AgentCore Runtime デプロイを設定します。また、スターターキットを設定して、起動時に Amazon ECR リポジトリを自動作成します。

設定ステップ中に、アプリケーションコードに基づいて Docker ファイルが生成されます

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude_context.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_claude_context"
)

### コンテキスト対応エージェントを AgentCore Runtime に起動

Docker ファイルができたので、コンテキスト対応エージェントを AgentCore Runtime に起動しましょう。これにより、Amazon ECR リポジトリと AgentCore Runtime が作成されます。

エージェントは、AgentCore Runtime がセッションを管理し、エージェントにコンテキスト情報を提供する方法を示します。

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

### AgentCore Runtime のステータス確認
AgentCore Runtime をデプロイしたので、デプロイステータスを確認しましょう

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

## セッション管理とコンテキスト処理の実演

異なるシナリオをテストして、AgentCore Runtime の主要なセッション管理機能を実演しましょう：

### シナリオ1: セッション継続性
同じセッション ID を複数の呼び出しに使用して、コンテキストがどのように維持されるかを示します。

### シナリオ2: セッション分離
異なるセッション ID を使用して、セッション間の完全な分離を実演します。

### シナリオ3: コンテキスト情報へのアクセス
エージェントがランタイムコンテキスト情報にアクセスする方法を示します。

<div style="text-align:left">
    <img src="images/invoke.png" width="85%"/>
</div>

In [ ]:
import uuid
import json
from IPython.display import Markdown, display

# Create a session ID for demonstrating session continuity
session_id = uuid.uuid4()
print(f"📋 Starting Session 1: {session_id}")
print(f"👤 User: Maira (ID: 1)")
print(f"❓ First question about weather\n")

invoke_response = agentcore_runtime.invoke({
    "prompt": "How is the weather outside?",
    "user_id": "1"
}, session_id=str(session_id))

response_data = invoke_response['response'][0]
display(Markdown(response_data))

In [ ]:
# Continue with the same session ID to demonstrate session continuity
print(f"🔄 Continuing Session 1: {session_id}")
print(f"👤 Same user: Maira (ID: 1)")
print(f"❓ Follow-up question about math\n")

invoke_response = agentcore_runtime.invoke({
    "prompt": "How much is 2X5?",
    "user_id": "1"
}, session_id=str(session_id))

response_data = invoke_response['response'][0]
display(Markdown(response_data))

In [ ]:
# Continue with the same session ID - notice how the agent remembers the previous calculation
print(f"🔄 Continuing Session 1: {session_id}")
print(f"👤 Same user: Maira (ID: 1)")
print(f"❓ Building on previous answer - demonstrates context continuity\n")

invoke_response = agentcore_runtime.invoke({
    "prompt": "and that plus 34?",
    "user_id": "1"
}, session_id=str(session_id))

response_data = invoke_response['response'][0]
display(Markdown(response_data))

In [ ]:
# NEW SESSION - Demonstrate session isolation
# Create a completely new session ID to show that context is lost
new_session_id = uuid.uuid4()
print(f"🆕 Starting NEW Session 2: {new_session_id}")
print(f"👤 Same user: Maira (ID: 1)")
print(f"❓ Attempting to reference previous calculation - should fail due to session isolation\n")

invoke_response = agentcore_runtime.invoke({
    "prompt": "And plus 10?",
    "user_id": "1"
}, session_id=str(new_session_id))

response_data = invoke_response['response'][0]
display(Markdown(response_data))

In [ ]:
# NEW SESSION AND USER - Demonstrate complete isolation
different_user_session = uuid.uuid4()
print(f"🆕 Starting Session 3: {different_user_session}")
print(f"👤 Different user: Mani (ID: 2)")
print(f"❓ Same question as first user - demonstrates user isolation\n")

invoke_response = agentcore_runtime.invoke({
    "prompt": "How is the weather?",
    "user_id": "2"
}, session_id=str(different_user_session))

response_data = invoke_response['response'][0]
display(Markdown(response_data))

## セッション管理結果の理解

上記の実演は、AgentCore Runtime のセッション管理のいくつかの重要な側面を示しています：

### 1. セッション継続性（セッション1）
- **最初の呼び出し**: エージェントが天気の質問に応答し、ユーザー名を認識
- **2回目の呼び出し**: エージェントが計算を実行（2×5=10）
- **3回目の呼び出し**: エージェントが以前の結果を参照（「それに34を足す」= 44）

**重要な学習**: エージェントは同じセッション内の複数の呼び出しにわたってコンテキストを維持し、以前のインタラクションからの計算結果を記憶していました。

### 2. セッション分離（セッション2）
- **新しいセッション ID**: 完全に新しいセッションを作成
- **同じユーザー**: 同じユーザー ID を使用したが、異なるセッション
- **コンテキストの喪失**: エージェントが以前の計算を参照できない

**重要な学習**: 同じユーザーでも、新しいセッションは以前のコンテキストにアクセスできない完全に分離された環境を作成します。

### 3. ユーザーとセッションの分離（セッション3）
- **異なるユーザー**: Maira ではなく Mani
- **新しいセッション**: 以前のセッションから完全に分離
- **新しいコンテキスト**: エージェントがクリーンな状態で開始

**重要な学習**: 各セッションは完全な分離を提供し、異なるユーザーとインタラクション間のプライバシーとセキュリティを確保します。

### 4. コンテキストオブジェクトの使用
すべての呼び出しを通じて、エージェントは：
- `context.session_id` を通じてランタイムコンテキストにアクセス
- カスタムペイロードデータ（`user_id`、`prompt`）を処理
- ログとデバッグ情報を維持

**重要な学習**: コンテキストオブジェクトは、エージェントが機能強化とデバッグに使用できる貴重なランタイム情報を提供します。

### 実演されたセッション管理のベストプラクティス

1. **会話の継続性のために一貫したセッション ID を使用**
2. **異なるユーザーや会話に対して一意のセッション ID を生成**
3. **エージェントの動作強化のためにコンテキスト情報を活用**
4. **セッション境界を設計** - セッション間の永続化を想定しない
5. **セッションが変更または期限切れになった場合の適切なコンテキスト損失の処理**

## クリーンアップ（オプション）

作成した AgentCore Runtime をクリーンアップしましょう

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# おめでとうございます！

Amazon Bedrock AgentCore Runtime を使用してセッション管理とコンテキスト処理を正常に実装し、テストしました！

## 学習した内容：

### セッション管理の基礎
* **セッション継続性**: 同じセッション ID が複数の呼び出しにわたってコンテキストを維持する方法
* **セッション分離**: 異なるセッション ID が完全に分離された環境を作成する方法
* **コンテキストの保持**: エージェントが状態を維持し、以前のインタラクションを参照する方法
* **セキュリティ境界**: AgentCore Runtime がユーザー間の完全な分離を確保する方法

### ランタイムコンテキスト処理
* **コンテキストオブジェクトへのアクセス**: `context` パラメータを通じてランタイム情報にアクセスする方法
* **セッション情報**: エージェントロジックでセッション ID を取得して使用する方法
* **ペイロード処理**: カスタムペイロードを通じて渡される構造化データを処理する方法
* **ランタイムメタデータ**: エージェントが実行環境情報にアクセスする方法

### AgentCore Runtime アーキテクチャ
* **MicroVM 分離**: 各セッションが独自の分離された microVM で実行される
* **リソース管理**: セッションごとの専用 CPU、メモリ、ファイルシステム
* **セキュリティモデル**: セッション終了後の完全なメモリサニタイゼーション
* **ライフサイクル管理**: セッション状態（アクティブ、アイドル、終了）とタイムアウト

### ベストプラクティスの実装
* **セッション ID 生成**: 異なる会話に対して一意の識別子を作成
* **コンテキストの活用**: エージェントの動作強化のためにランタイムコンテキストを活用
* **状態管理**: 一時的な状態と永続的な状態の理解
* **エラーハンドリング**: コンテキスト損失とセッション境界の適切な処理